# 02 - Transformer Architecture: Encoder-Decoder

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Implement **sinusoidal positional encoding** and visualize its properties
- Build a **Transformer encoder block** (Multi-Head Attention + Add & Norm + FFN + Add & Norm)
- Build a **Transformer decoder block** (Masked MHA + Cross-Attention + FFN, each with Add & Norm)
- Explain **layer normalization**, **residual connections**, and **feed-forward networks**
- Stack blocks into a full encoder and decoder
- Understand the complete Transformer architecture

## Prerequisites

- Completed Notebook 01 (self-attention, multi-head attention)
- Comfortable with `nn.Module`, `nn.Linear`, `nn.LayerNorm`
- Understanding of residual connections (ResNet intuition helpful)

## Table of Contents

1. [Positional Encoding](#1-positional-encoding)
2. [Layer Normalization](#2-layer-normalization)
3. [Residual Connections](#3-residual-connections)
4. [Feed-Forward Network](#4-feed-forward-network)
5. [Encoder Block](#5-encoder-block)
6. [Decoder Block](#6-decoder-block)
7. [Full Transformer Architecture](#7-full-transformer-architecture)
8. [Code: Implement Positional Encoding](#8-code-implement-positional-encoding)
9. [Code: Implement Encoder Block and Stack](#9-code-implement-encoder-block-and-stack)
10. [Code: Implement Decoder Block and Stack](#10-code-implement-decoder-block-and-stack)
11. [Exercise: Modify the Architecture](#11-exercise-modify-the-architecture)
12. [Common Mistakes & Debugging Tips](#12-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

print(f"PyTorch version: {torch.__version__}")

---
## 1. Positional Encoding

**Problem:** Self-attention is **permutation-invariant** -- it treats `[A, B, C]` the same as `[C, A, B]`. But word order matters in language.

**Solution:** Add **positional encoding** to the input embeddings to inject position information.

The original Transformer uses **sinusoidal positional encoding**:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

where:
- $pos$ = position in the sequence (0, 1, 2, ...)
- $i$ = dimension index
- $d_{\text{model}}$ = embedding dimension

**Why sinusoidal?**
- Each dimension oscillates at a different frequency (low dimensions = low freq, high dimensions = high freq)
- The model can learn to attend to **relative positions** because $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$
- Generalizes to longer sequences than seen during training

---
## 2. Layer Normalization

**Layer Normalization** normalizes across the feature dimension (not the batch dimension like BatchNorm):

$$\text{LN}(x) = \frac{x - \mu}{\sigma} \cdot \gamma + \beta$$

where:
- $\mu = \frac{1}{d} \sum_{i=1}^{d} x_i$ -- mean across features
- $\sigma = \sqrt{\frac{1}{d} \sum_{i=1}^{d} (x_i - \mu)^2 + \epsilon}$ -- std across features
- $\gamma, \beta$ -- learnable scale and shift parameters

**Why LayerNorm (not BatchNorm)?**
- Works with **variable-length sequences** (BatchNorm statistics depend on batch dimension)
- Does not depend on batch size -- works identically during training and inference
- Stabilizes training of deep Transformer stacks

In [ ]:
# LayerNorm demonstration
set_global_seed(42)

d_model = 8
x = torch.randn(2, 4, d_model)  # (batch, seq, features)

# Manual LayerNorm
mean = x.mean(dim=-1, keepdim=True)
std = x.std(dim=-1, keepdim=True, unbiased=False)
eps = 1e-5
x_norm_manual = (x - mean) / (std + eps)

# PyTorch LayerNorm (without learnable params for comparison)
ln = nn.LayerNorm(d_model, elementwise_affine=False)
x_norm_pytorch = ln(x)

print("Manual vs PyTorch LayerNorm match:", torch.allclose(x_norm_manual, x_norm_pytorch, atol=1e-5))
print(f"\nBefore LN - mean: {x[0,0].mean():.4f}, std: {x[0,0].std(unbiased=False):.4f}")
print(f"After LN  - mean: {x_norm_manual[0,0].mean():.4f}, std: {x_norm_manual[0,0].std(unbiased=False):.4f}")

---
## 3. Residual Connections

Each sub-layer in the Transformer uses a **residual connection**:

$$\text{output} = \text{LayerNorm}(x + \text{SubLayer}(x))$$

**Why residual connections?**
- **Gradient flow:** In deep networks, gradients can vanish through many layers. The residual shortcut provides a direct path for gradients.
- **Identity mapping:** The sublayer only needs to learn the **residual** (the difference from the input), which is easier to optimize.
- **Stability:** Prevents outputs from drifting too far from the input, especially early in training.

**Note:** The original Transformer paper uses **Post-Norm** (norm after residual addition). Some implementations use **Pre-Norm** (norm before sublayer). We use Post-Norm here to match the original paper.

In [ ]:
# Residual connection illustration
set_global_seed(42)

d_model = 8
x = torch.randn(1, 4, d_model)

# Some sublayer (e.g., a simple linear transform)
sublayer = nn.Linear(d_model, d_model)
norm = nn.LayerNorm(d_model)

# Without residual
out_no_residual = norm(sublayer(x))

# With residual (Post-Norm)
out_with_residual = norm(x + sublayer(x))

print(f"Input norm:              {x.norm():.4f}")
print(f"Without residual norm:   {out_no_residual.norm():.4f}")
print(f"With residual norm:      {out_with_residual.norm():.4f}")
print()
print("The residual connection keeps the output closer to the input's scale.")

---
## 4. Feed-Forward Network

Each Transformer block contains a **position-wise feed-forward network** (applied independently to each position):

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

Or with GELU activation (used in BERT, GPT):

$$\text{FFN}(x) = \text{GELU}(xW_1 + b_1)W_2 + b_2$$

**Key details:**
- Inner dimension is typically $4 \times d_{\text{model}}$ (e.g., $d_{\text{model}} = 512 \rightarrow d_{ff} = 2048$)
- This expansion-contraction pattern acts as a **bottleneck** that processes information
- Applied **identically** at each position (no cross-position interaction -- that is the attention layer's job)

In [ ]:
class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network."""
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == 'relu' else F.gelu
    
    def forward(self, x):
        # x: (batch, seq, d_model)
        return self.linear2(self.dropout(self.activation(self.linear1(x))))


# Test
set_global_seed(42)
d_model, d_ff = 16, 64  # 4x expansion
ffn = FeedForward(d_model, d_ff)
x = torch.randn(2, 6, d_model)
out = ffn(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"FFN params:   {sum(p.numel() for p in ffn.parameters())}")
print(f"  linear1: {d_model} x {d_ff} + {d_ff} = {d_model * d_ff + d_ff}")
print(f"  linear2: {d_ff} x {d_model} + {d_model} = {d_ff * d_model + d_model}")

---
## 5. Encoder Block

A single **Transformer encoder block** consists of:

```
Input
  |----+
  v    |
Multi-Head Self-Attention
  |    |
  v    |
 Add <-+  (residual)
  |
  v
LayerNorm
  |----+
  v    |
Feed-Forward Network
  |    |
  v    |
 Add <-+  (residual)
  |
  v
LayerNorm
  |
  v
Output
```

The encoder processes the **entire input sequence** at once (no causal masking).

---
## 6. Decoder Block

A single **Transformer decoder block** has three sub-layers:

```
Input (shifted right)
  |----+
  v    |
Masked Multi-Head Self-Attention   <-- causal mask (no future peeking)
  |    |
  v    |
 Add <-+
  |
  v
LayerNorm
  |----+
  v    |              Encoder Output (K, V)
Multi-Head Cross-Attention  -------+
  |    |                           |
  v    |                           |
 Add <-+
  |
  v
LayerNorm
  |----+
  v    |
Feed-Forward Network
  |    |
  v    |
 Add <-+
  |
  v
LayerNorm
  |
  v
Output
```

**Key difference from encoder:**
- **Masked self-attention:** Causal mask prevents attending to future tokens
- **Cross-attention:** Queries come from decoder, keys and values come from encoder output

---
## 7. Full Transformer Architecture

```
              ENCODER                              DECODER
         +--------------+                    +--------------+
         |   Input      |                    |   Output     |
         | Embedding    |                    | Embedding    |
         |    +         |                    |    +         |
         | Positional   |                    | Positional   |
         | Encoding     |                    | Encoding     |
         +------+-------+                    +------+-------+
                |                                   |
         +------v-------+                    +------v-------+
    Nx   | Encoder      |            Nx      | Decoder      |
    |    | Block        +---(K,V)--->        | Block        |
    |    +--------------+                    +--------------+
         |              |                    |              |
         +------+-------+                    +------+-------+
                                                    |
                                             +------v-------+
                                             |   Linear     |
                                             |   Softmax    |
                                             +--------------+
                                             | Output Probs |
                                             +--------------+
```

**Components:**
- **Input/Output Embeddings:** Convert token IDs to dense vectors
- **Positional Encoding:** Added to embeddings
- **N Encoder Blocks:** Process the full source sequence
- **N Decoder Blocks:** Generate the target sequence autoregressively
- **Linear + Softmax:** Project decoder output to vocabulary probabilities

---
## 8. Code: Implement Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding.
    
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)
        
        # Compute the div_term: 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )  # (d_model/2,)
        
        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd indices
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model) for broadcasting
        self.register_buffer('pe', pe)  # not a trainable parameter
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            x + positional encoding, same shape
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# Test
set_global_seed(42)
d_model = 64
pe = PositionalEncoding(d_model, max_len=100, dropout=0.0)

x = torch.zeros(1, 50, d_model)  # dummy input
encoded = pe(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {encoded.shape}")
print(f"PE is added, not concatenated (same shape).")

In [ ]:
# Visualize positional encoding
d_model = 64
max_len = 100
pe_module = PositionalEncoding(d_model, max_len=max_len, dropout=0.0)

# Extract the PE matrix
pe_values = pe_module.pe[0, :max_len, :].numpy()  # (max_len, d_model)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap of all PE values
im = axes[0].imshow(pe_values, aspect='auto', cmap='RdBu_r', interpolation='nearest')
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Position')
axes[0].set_title('Positional Encoding Heatmap')
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

# Plot specific dimensions
dims_to_plot = [0, 1, 2, 3, 10, 20]
for d in dims_to_plot:
    axes[1].plot(pe_values[:, d], label=f'dim {d}')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('PE Value')
axes[1].set_title('Positional Encoding by Dimension')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Lower dimensions have higher frequency (change rapidly with position).")
print("Higher dimensions have lower frequency (change slowly with position).")

In [ ]:
# Demonstrate that PE allows relative position representation
# PE(pos+k) can be expressed as a linear function of PE(pos)
# Check: dot product similarity between PE vectors at different distances

pe_vectors = pe_module.pe[0].numpy()  # (max_len, d_model)

# Compute cosine similarity between positions
from numpy.linalg import norm

n_positions = 50
similarity_matrix = np.zeros((n_positions, n_positions))
for i in range(n_positions):
    for j in range(n_positions):
        similarity_matrix[i, j] = np.dot(pe_vectors[i], pe_vectors[j]) / (
            norm(pe_vectors[i]) * norm(pe_vectors[j]))

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(similarity_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('Position')
ax.set_ylabel('Position')
ax.set_title('Cosine Similarity Between Positional Encodings')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print("Nearby positions have higher similarity; distant positions are less similar.")
print("The pattern is based on relative distance, enabling the model to learn relative positions.")

---
## 9. Code: Implement Encoder Block and Stack

We reuse the `MultiHeadAttention` from Notebook 01 and build the full encoder.

In [ ]:
# Bring in scaled_dot_product_attention and MultiHeadAttention from Notebook 01

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention."""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights


class MultiHeadAttention(nn.Module):
    """Multi-Head Attention."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        Q = self.W_Q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_O(attn_output)
        return output, attn_weights


print("MultiHeadAttention and FeedForward modules ready.")

In [ ]:
class EncoderBlock(nn.Module):
    """
    Single Transformer Encoder Block.
    
    Structure: Multi-Head Attention -> Add & Norm -> FFN -> Add & Norm
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: Optional attention mask
        Returns:
            output: (batch, seq_len, d_model)
        """
        # Sub-layer 1: Multi-Head Self-Attention + Add & Norm
        attn_output, _ = self.self_attn(x, x, x, mask=mask)  # Q=K=V=x
        x = self.norm1(x + self.dropout1(attn_output))
        
        # Sub-layer 2: FFN + Add & Norm
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_output))
        
        return x


# Test single encoder block
set_global_seed(42)
d_model, n_heads, d_ff = 16, 4, 64
enc_block = EncoderBlock(d_model, n_heads, d_ff)
x = torch.randn(2, 6, d_model)
out = enc_block(x)
print(f"Encoder block input:  {x.shape}")
print(f"Encoder block output: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in enc_block.parameters())}")

In [ ]:
class TransformerEncoder(nn.Module):
    """
    Full Transformer Encoder: Embedding + Positional Encoding + N Encoder Blocks.
    """
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers,
                 max_len=5000, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)  # final norm
    
    def forward(self, src, mask=None):
        """
        Args:
            src: (batch, seq_len) -- token IDs
            mask: Optional attention mask
        Returns:
            output: (batch, seq_len, d_model)
        """
        # Embed and scale
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        # Pass through encoder blocks
        for layer in self.layers:
            x = layer(x, mask=mask)
        
        return self.norm(x)


# Test full encoder
set_global_seed(42)
encoder = TransformerEncoder(
    vocab_size=1000, d_model=32, n_heads=4, d_ff=128,
    n_layers=3, dropout=0.1
)

src_tokens = torch.randint(0, 1000, (2, 10))  # batch=2, seq=10
enc_output = encoder(src_tokens)

print(f"Source tokens shape:  {src_tokens.shape}")
print(f"Encoder output shape: {enc_output.shape}")
print(f"Total parameters:     {sum(p.numel() for p in encoder.parameters()):,}")

---
## 10. Code: Implement Decoder Block and Stack

In [ ]:
class DecoderBlock(nn.Module):
    """
    Single Transformer Decoder Block.
    
    Structure:
      Masked Self-Attention -> Add & Norm ->
      Cross-Attention -> Add & Norm ->
      FFN -> Add & Norm
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        """
        Args:
            x: (batch, tgt_seq_len, d_model) -- decoder input
            enc_output: (batch, src_seq_len, d_model) -- encoder output
            src_mask: mask for encoder output (padding)
            tgt_mask: causal mask for decoder self-attention
        Returns:
            output: (batch, tgt_seq_len, d_model)
        """
        # Sub-layer 1: Masked Multi-Head Self-Attention
        attn_output, _ = self.masked_self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout1(attn_output))
        
        # Sub-layer 2: Cross-Attention (Q from decoder, K,V from encoder)
        cross_output, _ = self.cross_attn(x, enc_output, enc_output, mask=src_mask)
        x = self.norm2(x + self.dropout2(cross_output))
        
        # Sub-layer 3: FFN
        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_output))
        
        return x


# Test single decoder block
set_global_seed(42)
d_model, n_heads, d_ff = 16, 4, 64
dec_block = DecoderBlock(d_model, n_heads, d_ff)

tgt = torch.randn(2, 5, d_model)   # decoder input
enc_out = torch.randn(2, 8, d_model)  # encoder output

# Create causal mask for target
tgt_len = 5
tgt_mask = torch.tril(torch.ones(tgt_len, tgt_len)).bool().unsqueeze(0).unsqueeze(0)

dec_out = dec_block(tgt, enc_out, tgt_mask=tgt_mask)
print(f"Decoder input shape:  {tgt.shape}")
print(f"Encoder output shape: {enc_out.shape}")
print(f"Decoder output shape: {dec_out.shape}")
print(f"Parameters: {sum(p.numel() for p in dec_block.parameters())}")

In [ ]:
class TransformerDecoder(nn.Module):
    """
    Full Transformer Decoder: Embedding + Positional Encoding + N Decoder Blocks.
    """
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers,
                 max_len=5000, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        """
        Args:
            tgt: (batch, tgt_seq_len) -- target token IDs
            enc_output: (batch, src_seq_len, d_model)
            src_mask: padding mask for source
            tgt_mask: causal mask for target
        Returns:
            output: (batch, tgt_seq_len, d_model)
        """
        x = self.embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        for layer in self.layers:
            x = layer(x, enc_output, src_mask=src_mask, tgt_mask=tgt_mask)
        
        return self.norm(x)


class Transformer(nn.Module):
    """
    Full Transformer: Encoder + Decoder + Output Projection.
    """
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=64,
                 n_heads=4, d_ff=256, n_layers=3, max_len=5000, dropout=0.1):
        super().__init__()
        self.encoder = TransformerEncoder(
            src_vocab_size, d_model, n_heads, d_ff, n_layers, max_len, dropout
        )
        self.decoder = TransformerDecoder(
            tgt_vocab_size, d_model, n_heads, d_ff, n_layers, max_len, dropout
        )
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encoder(src, mask=src_mask)
        dec_output = self.decoder(tgt, enc_output, src_mask=src_mask, tgt_mask=tgt_mask)
        logits = self.output_projection(dec_output)  # (batch, tgt_seq, tgt_vocab)
        return logits


# Test full Transformer
set_global_seed(42)
model = Transformer(
    src_vocab_size=500, tgt_vocab_size=500,
    d_model=32, n_heads=4, d_ff=128, n_layers=2
)

src = torch.randint(0, 500, (2, 8))   # source tokens
tgt = torch.randint(0, 500, (2, 6))   # target tokens

# Create causal mask for target
tgt_len = tgt.size(1)
tgt_mask = torch.tril(torch.ones(tgt_len, tgt_len)).bool().unsqueeze(0).unsqueeze(0)

logits = model(src, tgt, tgt_mask=tgt_mask)

print(f"Source shape:  {src.shape}")
print(f"Target shape:  {tgt.shape}")
print(f"Logits shape:  {logits.shape}  (batch, tgt_seq, vocab)")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print()
for name, param in model.named_parameters():
    if 'layers.0' in name or 'embedding' in name or 'output' in name:
        print(f"  {name}: {param.shape}")

---
## 11. Exercise: Modify the Architecture

**Task:** Experiment with different architectural configurations.

1. Create a Transformer with `d_model=64, n_heads=8, n_layers=4`
2. Create another with `d_model=64, n_heads=2, n_layers=6`
3. Compare the total parameter counts
4. Run a forward pass on both and verify output shapes

```python
# Your code here
# model_a = Transformer(src_vocab_size=500, tgt_vocab_size=500, ...)
# model_b = Transformer(src_vocab_size=500, tgt_vocab_size=500, ...)
# Compare parameter counts and run forward passes
```

In [ ]:
# Try the exercise yourself before looking at the solution!






### Solution

In [ ]:
# ----- Solution -----
set_global_seed(42)

# Configuration A: more heads, fewer layers
model_a = Transformer(
    src_vocab_size=500, tgt_vocab_size=500,
    d_model=64, n_heads=8, d_ff=256, n_layers=4
)

# Configuration B: fewer heads, more layers
model_b = Transformer(
    src_vocab_size=500, tgt_vocab_size=500,
    d_model=64, n_heads=2, d_ff=256, n_layers=6
)

params_a = sum(p.numel() for p in model_a.parameters())
params_b = sum(p.numel() for p in model_b.parameters())

print(f"Model A (8 heads, 4 layers): {params_a:,} parameters")
print(f"Model B (2 heads, 6 layers): {params_b:,} parameters")
print(f"Ratio B/A: {params_b/params_a:.2f}x")
print()

# Forward pass on both
src = torch.randint(0, 500, (2, 10))
tgt = torch.randint(0, 500, (2, 8))
tgt_len = tgt.size(1)
tgt_mask = torch.tril(torch.ones(tgt_len, tgt_len)).bool().unsqueeze(0).unsqueeze(0)

logits_a = model_a(src, tgt, tgt_mask=tgt_mask)
logits_b = model_b(src, tgt, tgt_mask=tgt_mask)

print(f"Model A output shape: {logits_a.shape}")
print(f"Model B output shape: {logits_b.shape}")
print()
print("Key insight: More layers = more parameters (each layer has MHA + FFN).")
print("Number of heads does not change parameter count (just splits d_model differently).")

---
## 12. Common Mistakes & Debugging Tips

**1. Forgetting positional encoding**
- Without PE, the model treats `[A, B, C]` and `[C, B, A]` identically
- Symptom: Model performance is much worse than expected, especially on tasks where order matters

**2. Scaling embeddings by $\sqrt{d_{\text{model}}}$**
- The original paper multiplies embeddings by $\sqrt{d_{\text{model}}}$ before adding PE
- This keeps the embedding magnitudes comparable to the PE magnitudes
- Forgetting this can cause the PE signal to be drowned out

**3. Wrong norm placement (Pre-Norm vs Post-Norm)**
- Post-Norm (original paper): `LayerNorm(x + SubLayer(x))`
- Pre-Norm (more stable): `x + SubLayer(LayerNorm(x))`
- Pre-Norm tends to train more stably for deeper models, but outputs need a final LayerNorm

**4. Cross-attention Q/K/V mix-up**
- In decoder cross-attention: Q = decoder, K = V = encoder output
- A common bug is swapping these, which produces incorrect results

**5. Missing causal mask in decoder**
- Without the causal mask, the decoder can cheat by looking at future tokens
- Training loss will be artificially low, but generation at inference will fail

**6. Not sharing embeddings / output projection weights**
- The original paper shares weights between the target embedding and the output projection
- This reduces parameters and can improve performance
- To share: `model.output_projection.weight = model.decoder.embedding.weight`

**7. Gradient explosion with deep stacks**
- Use learning rate warmup (linear warmup then decay)
- The original paper uses: $lr = d_{\text{model}}^{-0.5} \cdot \min(step^{-0.5}, step \cdot warmup^{-1.5})$
- Pre-Norm architecture also helps with stability

---

**Next notebook:** [03 - Training a Small Transformer for Text Classification](03_Training_a_Small_Transformer_Text_Classification.ipynb) -- we put it all together and train a Transformer encoder on a real text classification task.